In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

In [3]:
ticker = "HDFCBANK.NS"
start_date = "2015-01-01"
end_date = "2025-01-01"

In [4]:
df = pd.read_csv(DATA_PROCESSED / f"{ticker}_engineered_{start_date}_{end_date}.csv", index_col=0, parse_dates=True)

In [5]:
df['close_to_sma20'] = df['Close'] / df['sma_20'] - 1
df['close_to_sma50'] = df['Close'] / df['sma_50'] - 1
df['sma10_to_sma50'] = df['sma_10'] / df['sma_50'] - 1
df['bb_pct'] = (df['Close'] - df['bb_low']) / (df['bb_high'] - df['bb_low'])
df['macd_pct'] = df['macd'] / df['Close']

In [6]:
df['ret_5d'] = df['Close'].pct_change(5)
df['ret_10d'] = df['Close'].pct_change(10)
df['vol_20d'] = df['daily_return'].rolling(20).std()
df['volume_ratio'] = df['Volume'] / df['Volume'].rolling(20).mean()

In [7]:
old_features = [
    'Open', 'High', 'Low', 'Close', 'Volume', 'daily_return',
    'sma_10', 'sma_20', 'sma_50', 'ema_10', 'ema_20',
    'rsi_14', 'macd', 'macd_signal', 'macd_diff',
    'bb_high', 'bb_low', 'bb_width'
]

In [8]:
new_features = [
    'rsi_14', 'daily_return', 'bb_width',
    'close_to_sma20', 'close_to_sma50', 'sma10_to_sma50',
    'bb_pct', 'macd_pct', 'ret_5d', 'ret_10d', 'vol_20d', 'volume_ratio'
]

In [9]:
df = df.dropna()
print(df.shape)

(2398, 28)


In [10]:
cols_to_check = ['Close', 'sma_20', 'close_to_sma20', 'macd', 'macd_pct', 'volume_ratio']
df[cols_to_check].groupby(df.index.year).mean()

,Close,sma_20,close_to_sma20,macd,macd_pct,volume_ratio
Date,,,,,,
2015,237.681818,237.169795,0.002329,0.284351,0.000868,0.979733
2016,263.316475,262.225514,0.004397,0.890966,0.003071,1.004842
2017,372.011007,366.006448,0.017669,4.246285,0.011820,1.020604
2018,461.183157,458.849330,0.005224,1.570700,0.003190,1.015198
2019,540.196625,536.013926,0.007908,3.184121,0.005581,1.013528
2020,525.175102,522.907620,0.004672,1.472571,-0.000190,1.033814
2021,706.021612,704.846505,0.001937,1.723843,0.002099,0.978207
2022,689.010733,685.711916,0.004795,1.631409,0.001718,1.015987
2023,767.506038,766.380719,0.001569,0.191018,-0.000026,1.061127


In [11]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

In [12]:
y = df['target']
feature_sets = {"old_18": old_features, "new_12": new_features}
models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42)
}
tscv = TimeSeriesSplit(n_splits=5)
results = []

In [13]:
for fs_name, cols in feature_sets.items():
    X = df[cols]
    for model_name, model in models.items():
        for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
            model.fit(X.iloc[train_idx], y.iloc[train_idx])
            preds = model.predict(X.iloc[test_idx])
            probs = model.predict_proba(X.iloc[test_idx])[:, 1]
            results.append({
                "feature_set": fs_name,
                "model": model_name,
                "fold": fold,
                "accuracy": accuracy_score(y.iloc[test_idx], preds),
                "recall": recall_score(y.iloc[test_idx], preds),
                "auc": roc_auc_score(y.iloc[test_idx], probs)
            })

In [14]:
results_df = pd.DataFrame(results)
results_df.groupby(["feature_set", "model"])[["accuracy", "recall", "auc"]].mean()

accuracy    recall       auc
feature_set model                                     
new_12      RandomForest  0.481203  0.475810  0.493264
            XGBoost       0.495238  0.492721  0.504132
old_18      RandomForest  0.498747  0.386667  0.511662
            XGBoost       0.510276  0.437337  0.526652

In [15]:
results_df.pivot_table(index="fold", columns=["feature_set", "model"], values="auc")

feature_set       new_12                 old_18          
model       RandomForest   XGBoost RandomForest   XGBoost
fold                                                     
1               0.477198  0.509847     0.472147  0.522994
2               0.512100  0.510579     0.509926  0.518872
3               0.482255  0.472836     0.533792  0.533754
4               0.505757  0.501612     0.553729  0.541194
5               0.489008  0.525787     0.488718  0.516444

In [16]:
df.to_csv(DATA_PROCESSED / f"{ticker}_stationary_{start_date}_{end_date}.csv")